# Redes Neuronales: TensorFlow vs Scikit-Learn

## Introducción

Este notebook compara la implementación de redes neuronales usando:
- **Scikit-Learn (MLPClassifier/MLPRegressor)**: Ideal para prototipos rápidos y datasets pequeños
- **TensorFlow/Keras**: Framework completo para deep learning con control total

### Diferencias Clave

| Aspecto | Scikit-Learn | TensorFlow/Keras |
|---------|--------------|------------------|
| **Complejidad** | Simple, pocas líneas | Más código, más control |
| **Flexibilidad** | Limitada | Total |
| **GPU** | No soporta | Sí soporta |
| **Arquitecturas** | Solo MLP básico | CNN, RNN, Transformers, etc. |
| **Callbacks** | Básico (early_stopping) | Completos (EarlyStopping, ModelCheckpoint, TensorBoard...) |
| **Personalización** | Mínima | Capas, optimizadores, pérdida custom |
| **Uso recomendado** | Prototipos, datasets pequeños | Producción, deep learning |

In [ ]:
# Si no tenemos instalado tesorflow, descomenta la siguiente línea para instalarlo
%pip install tensorflow

In [ ]:
# Importar librerías
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_circles
from sklearn.metrics import accuracy_score, classification_report

# TensorFlow/Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks

print(f"TensorFlow versión: {tf.__version__}")
print(f"Keras versión: {keras.__version__}")

# Configuración
np.random.seed(42)
tf.random.set_seed(42)

## 1. Crear Dataset de Ejemplo

Usaremos el dataset "moons" (lunas) que es no linealmente separable, similar al problema XOR.

In [ ]:
# Crear dataset de lunas (no linealmente separable)
X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

# Dividir en entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalizar los datos (MUY IMPORTANTE para redes neuronales)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Visualizar
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c='blue', label='Clase 0', alpha=0.6)
plt.scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c='red', label='Clase 1', alpha=0.6)
plt.title('Datos Originales')
plt.xlabel('X1')
plt.ylabel('X2')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(X_train_scaled[y_train == 0, 0], X_train_scaled[y_train == 0, 1], c='blue', label='Clase 0', alpha=0.6)
plt.scatter(X_train_scaled[y_train == 1, 0], X_train_scaled[y_train == 1, 1], c='red', label='Clase 1', alpha=0.6)
plt.title('Datos Normalizados (StandardScaler)')
plt.xlabel('X1 (normalizado)')
plt.ylabel('X2 (normalizado)')
plt.legend()

plt.tight_layout()
plt.show()

print(f"Tamaño del dataset: {len(X)} muestras")
print(f"Entrenamiento: {len(X_train)} | Test: {len(X_test)}")

---
## 2. Implementación con Scikit-Learn (MLPClassifier)

### Características de MLPClassifier:
- **Simplicidad**: API consistente con otros modelos de sklearn (fit/predict)
- **Parámetros limitados**: Solo puedes configurar capas ocultas básicas
- **Sin GPU**: Solo CPU
- **Ideal para**: Prototipos rápidos, comparaciones, datasets pequeños

In [ ]:
# =====================================================
# IMPLEMENTACIÓN CON SCIKIT-LEARN
# =====================================================

# Crear el modelo MLP
mlp_sklearn = MLPClassifier(
    hidden_layer_sizes=(16, 8),    # 2 capas ocultas: 16 y 8 neuronas
    activation='relu',              # Función de activación
    solver='adam',                  # Optimizador
    alpha=0.0001,                   # Regularización L2
    learning_rate_init=0.001,       # Learning rate inicial
    max_iter=500,                   # Máximo de épocas
    random_state=42,
    verbose=False                   # No mostrar progreso
)

# Entrenar (¡solo una línea!)
mlp_sklearn.fit(X_train_scaled, y_train)

# Predecir
y_pred_sklearn = mlp_sklearn.predict(X_test_scaled)

# Evaluar
acc_sklearn = accuracy_score(y_test, y_pred_sklearn)

print("=" * 60)
print("RESULTADOS SCIKIT-LEARN (MLPClassifier)")
print("=" * 60)
print(f"\n📊 Arquitectura: entrada(2) → 16 → 8 → salida(1)")
print(f"📊 Activación: {mlp_sklearn.activation}")
print(f"📊 Épocas ejecutadas: {mlp_sklearn.n_iter_}")
print(f"\n✅ Precisión en Test: {acc_sklearn*100:.2f}%")

---
## 3. Implementación con TensorFlow/Keras

### Características de TensorFlow:
- **Control total**: Define cada capa, optimizador, función de pérdida
- **GPU**: Aprovecha tarjetas gráficas para entrenamiento rápido
- **Callbacks**: EarlyStopping, ModelCheckpoint, TensorBoard, etc.
- **Arquitecturas avanzadas**: CNN, RNN, LSTM, Transformers
- **Ideal para**: Producción, deep learning, modelos complejos

### Conceptos clave en TensorFlow/Keras:
1. **Sequential**: Modelo donde las capas van una tras otra
2. **Dense**: Capa completamente conectada (igual que en sklearn)
3. **compile()**: Define optimizador, función de pérdida y métricas
4. **fit()**: Entrena el modelo (similar a sklearn pero con más opciones)
5. **evaluate()**: Evalúa en datos de test

In [ ]:
# =====================================================
# IMPLEMENTACIÓN CON TENSORFLOW/KERAS
# =====================================================

# PASO 1: Definir la arquitectura del modelo
# ------------------------------------------
model_tf = keras.Sequential([
    # Capa de entrada: especificamos la forma de los datos
    layers.Input(shape=(2,)),  # 2 features de entrada
    
    # Primera capa oculta: 16 neuronas con activación ReLU
    layers.Dense(16, activation='relu', name='capa_oculta_1'),
    
    # Segunda capa oculta: 8 neuronas con activación ReLU
    layers.Dense(8, activation='relu', name='capa_oculta_2'),
    
    # Capa de salida: 1 neurona con sigmoide (clasificación binaria)
    layers.Dense(1, activation='sigmoid', name='salida')
])

# Ver resumen del modelo
print("=" * 60)
print("ARQUITECTURA DEL MODELO TENSORFLOW/KERAS")
print("=" * 60)
model_tf.summary()

In [ ]:
# PASO 2: Compilar el modelo
# --------------------------
# Aquí definimos:
# - optimizer: algoritmo de optimización (Adam, SGD, RMSprop, etc.)
# - loss: función de pérdida a minimizar
# - metrics: métricas a monitorear durante el entrenamiento

model_tf.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',  # Para clasificación binaria
    metrics=['accuracy']
)

print("✅ Modelo compilado con:")
print("   - Optimizador: Adam (lr=0.001)")
print("   - Pérdida: Binary Crossentropy")
print("   - Métricas: Accuracy")

In [ ]:
# PASO 3: Entrenar el modelo
# --------------------------
# Configurar callbacks (en sklearn existe early_stopping=True, pero TensorFlow es más flexible)

# EarlyStopping: detiene el entrenamiento si no mejora
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',      # Monitorea la pérdida en validación
    patience=20,             # Espera 20 épocas sin mejora antes de parar
    restore_best_weights=True  # Restaura los mejores pesos
)

# Entrenar el modelo
print("Entrenando modelo TensorFlow...")
history = model_tf.fit(
    X_train_scaled, y_train,
    epochs=200,                          # Máximo de épocas
    batch_size=32,                       # Tamaño de batch (bach size se usa para actualizar pesos cada 32 muestras)
    validation_split=0.2,                # 20% para validación
    callbacks=[early_stop],              # Callbacks
    verbose=0                            # No mostrar progreso por época
)

print(f"✅ Entrenamiento completado en {len(history.history['loss'])} épocas")

In [ ]:
# PASO 4: Visualizar el entrenamiento (¡Esto no existe en sklearn!)
# ----------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva de pérdida
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida (Binary Crossentropy)')
axes[0].set_title('Curva de Pérdida durante el Entrenamiento')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Curva de precisión
axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Precisión')
axes[1].set_title('Curva de Precisión durante el Entrenamiento')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('📈 Monitoreo del Entrenamiento (Ventaja de TensorFlow)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n💡 Estas curvas permiten detectar:")
print("   - Overfitting: si val_loss sube mientras train_loss baja")
print("   - Underfitting: si ambas pérdidas son altas")
print("   - Convergencia: cuando las curvas se estabilizan")

In [ ]:
# PASO 5: Evaluar el modelo
# -------------------------
loss_tf, acc_tf = model_tf.evaluate(X_test_scaled, y_test, verbose=0)

print("=" * 60)
print("RESULTADOS TENSORFLOW/KERAS")
print("=" * 60)
print(f"\n📊 Arquitectura: entrada(2) → 16 → 8 → salida(1)")
print(f"📊 Épocas ejecutadas: {len(history.history['loss'])}")
print(f"📊 Pérdida en Test: {loss_tf:.4f}")
print(f"\n✅ Precisión en Test: {acc_tf*100:.2f}%")

---
## 4. Comparación Visual: Fronteras de Decisión

Veamos cómo cada implementación separa las clases:

In [ ]:
# Función para dibujar fronteras de decisión
from matplotlib.colors import ListedColormap

def plot_decision_boundary_comparison(X, y, scaler, model_sklearn, model_tf):
    """Compara las fronteras de decisión de sklearn vs TensorFlow"""
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Crear malla para las predicciones
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    # Escalar la malla
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid_points)
    
    cmap_light = ListedColormap(['#AAAAFF', '#FFAAAA'])
    cmap_bold = ListedColormap(['#0000FF', '#FF0000'])
    
    # --- Scikit-Learn ---
    Z_sklearn = model_sklearn.predict(grid_scaled).reshape(xx.shape)
    axes[0].contourf(xx, yy, Z_sklearn, alpha=0.4, cmap=cmap_light)
    axes[0].contour(xx, yy, Z_sklearn, colors='black', linewidths=2, linestyles='--')
    axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, edgecolor='black', s=50, alpha=0.7)
    axes[0].set_title(f'Scikit-Learn (MLPClassifier)\nPrecisión: {acc_sklearn*100:.2f}%', fontsize=12)
    axes[0].set_xlabel('X1')
    axes[0].set_ylabel('X2')
    
    # --- TensorFlow ---
    Z_tf = (model_tf.predict(grid_scaled, verbose=0) > 0.5).astype(int).reshape(xx.shape)
    axes[1].contourf(xx, yy, Z_tf, alpha=0.4, cmap=cmap_light)
    axes[1].contour(xx, yy, Z_tf, colors='black', linewidths=2, linestyles='--')
    axes[1].scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold, edgecolor='black', s=50, alpha=0.7)
    axes[1].set_title(f'TensorFlow/Keras\nPrecisión: {acc_tf*100:.2f}%', fontsize=12)
    axes[1].set_xlabel('X1')
    axes[1].set_ylabel('X2')
    
    plt.suptitle('Comparación de Fronteras de Decisión', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Visualizar
plot_decision_boundary_comparison(X_test, y_test, scaler, mlp_sklearn, model_tf)

---
## 5. Comparación de Código: Lado a Lado

### Scikit-Learn (5 líneas esenciales)
```python
# Crear modelo
model = MLPClassifier(hidden_layer_sizes=(16, 8), activation='relu')

# Entrenar
model.fit(X_train, y_train)

# Predecir
predictions = model.predict(X_test)
```

### TensorFlow/Keras (15+ líneas, más control)
```python
# Crear modelo (más explícito)
model = keras.Sequential([
    layers.Input(shape=(2,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

# Compilar (¡nuevo paso!)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Entrenar (más opciones)
history = model.fit(X_train, y_train, 
                    epochs=100, 
                    validation_split=0.2,
                    callbacks=[EarlyStopping()])

# Predecir
predictions = (model.predict(X_test) > 0.5).astype(int)
```

---
## 6. Ventajas Exclusivas de TensorFlow

### Lo que TensorFlow puede hacer y Scikit-Learn NO:

In [ ]:
# =====================================================
# VENTAJA 1: Regularización avanzada (Dropout, BatchNorm)
# =====================================================

model_avanzado = keras.Sequential([
    layers.Input(shape=(2,)),
    
    # Capa con Batch Normalization (normaliza las activaciones)
    layers.Dense(32, activation='relu'),
    layers.BatchNormalization(),  # ← NO existe en sklearn
    
    # Capa con Dropout (previene overfitting)
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.3),  # ← NO existe en sklearn (apaga 30% de neuronas)
    
    layers.Dense(8, activation='relu'),
    layers.Dropout(0.2),
    
    layers.Dense(1, activation='sigmoid')
])

model_avanzado.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("=" * 60)
print("MODELO CON REGULARIZACIÓN AVANZADA")
print("=" * 60)
model_avanzado.summary()

print("\n💡 Dropout y BatchNormalization son técnicas clave para:")
print("   - Prevenir overfitting")
print("   - Entrenar redes más profundas")
print("   - Mejorar la generalización")

In [ ]:
# =====================================================
# VENTAJA 2: Learning Rate Schedulers
# =====================================================
# Permite cambiar el learning rate durante el entrenamiento

# Ejemplo: reducir lr cuando la validación no mejora
lr_scheduler = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,       # Reduce lr a la mitad
    patience=5,       # Espera 5 épocas sin mejora
    min_lr=0.00001
)

# Ejemplo: Learning rate que decae con las épocas
def lr_schedule(epoch, lr):
    if epoch < 10:
        return lr
    else:
        return lr * 0.95  # Reduce 5% cada época después de la 10

lr_callback = callbacks.LearningRateScheduler(lr_schedule)

print("📉 Learning Rate Schedulers disponibles en TensorFlow:")
print("   - ReduceLROnPlateau: reduce lr cuando no mejora")
print("   - LearningRateScheduler: función custom de lr")
print("   - ExponentialDecay, CosineDecay, etc.")
print("\n⚠️ En sklearn: el learning rate es FIJO durante todo el entrenamiento")

In [ ]:
# =====================================================
# VENTAJA 3: Guardar y cargar modelos fácilmente
# =====================================================

# Guardar modelo completo (arquitectura + pesos + optimizador)
# model_tf.save('mi_modelo.keras')

# Cargar modelo
# modelo_cargado = keras.models.load_model('mi_modelo.keras')

# Guardar solo pesos
# model_tf.save_weights('pesos.weights.h5')

# ModelCheckpoint: guarda automáticamente el mejor modelo durante el entrenamiento
checkpoint = callbacks.ModelCheckpoint(
    'mejor_modelo.keras',
    monitor='val_accuracy',
    save_best_only=True,  # Solo guarda si mejora
    mode='max'
)

print("💾 Opciones de guardado en TensorFlow:")
print("   - model.save(): guarda modelo completo (.keras)")
print("   - model.save_weights(): solo los pesos")
print("   - ModelCheckpoint: guarda automáticamente durante entrenamiento")
print("\n📦 En sklearn: solo puedes usar pickle/joblib (menos robusto)")

---
## 7. Resumen: ¿Cuándo usar cada uno?

### 🔹 Usa **Scikit-Learn** cuando:
- ✅ Prototipos rápidos y experimentos iniciales
- ✅ Datasets pequeños (< 10,000 muestras)
- ✅ Quieres comparar MLP con otros modelos de sklearn (Random Forest, SVM, etc.)
- ✅ No necesitas GPU
- ✅ Simplicidad es prioritaria

### 🔸 Usa **TensorFlow/Keras** cuando:
- ✅ Producción y modelos en serio
- ✅ Datasets grandes (> 10,000 muestras)
- ✅ Necesitas GPU para acelerar entrenamiento
- ✅ Arquitecturas complejas (CNN, RNN, Transformers)
- ✅ Necesitas callbacks avanzados (ModelCheckpoint, TensorBoard, ReduceLROnPlateau, etc.)
- ✅ Regularización avanzada (Dropout, BatchNorm)
- ✅ Fine-tuning de learning rate
- ✅ Monitoreo detallado del entrenamiento
- ✅ Despliegue en producción (TensorFlow Serving, TFLite)

In [ ]:
# =====================================================
# COMPARACIÓN FINAL
# =====================================================

print("=" * 70)
print("COMPARACIÓN FINAL: MISMA ARQUITECTURA, DIFERENTES FRAMEWORKS")
print("=" * 70)

print(f"""
┌─────────────────────────┬─────────────────┬─────────────────┐
│        Aspecto          │  Scikit-Learn   │   TensorFlow    │
├─────────────────────────┼─────────────────┼─────────────────┤
│ Precisión en Test       │    {acc_sklearn*100:6.2f}%      │    {acc_tf*100:6.2f}%      │
├─────────────────────────┼─────────────────┼─────────────────┤
│ Líneas de código        │      ~5         │     ~15         │
├─────────────────────────┼─────────────────┼─────────────────┤
│ Curvas de aprendizaje   │      ❌         │      ✅         │
├─────────────────────────┼─────────────────┼─────────────────┤
│ Early Stopping          │   ✅ (básico)   │  ✅ (avanzado)  │
├─────────────────────────┼─────────────────┼─────────────────┤
│ Dropout/BatchNorm       │      ❌         │      ✅         │
├─────────────────────────┼─────────────────┼─────────────────┤
│ GPU Support             │      ❌         │      ✅         │
├─────────────────────────┼─────────────────┼─────────────────┤
│ Guardar modelo          │   pickle/joblib │  .keras/.h5     │
├─────────────────────────┼─────────────────┼─────────────────┤
│ Arquitecturas avanzadas │      ❌         │  CNN/RNN/etc    │
└─────────────────────────┴─────────────────┴─────────────────┘
""")

print("💡 CONCLUSIÓN:")
print("   • sklearn: Ideal para empezar y prototipos rápidos")
print("   • TensorFlow: Imprescindible para deep learning profesional")
print("   • Ambos pueden dar resultados similares en problemas simples")

---
## 8. Bonus: Problema XOR en TensorFlow

Para comparar directamente con el notebook anterior, resolvamos XOR con TensorFlow:

In [ ]:
# Datos XOR
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float32)
y_xor = np.array([[0], [1], [1], [0]], dtype=np.float32)

# Función para crear y entrenar el modelo XOR
# (La convergencia depende de la inicialización, igual que en sklearn)
def crear_modelo_xor():
    model = keras.Sequential([
        layers.Input(shape=(2,)),
        layers.Dense(4, activation='tanh'),  # 4 neuronas (más margen para converger)
        layers.Dense(1, activation='sigmoid')
    ])
    # Learning rate más alto (0.1) ayuda a converger en XOR
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.1),
        loss='binary_crossentropy', 
        metrics=['accuracy']
    )
    return model

# Entrenar hasta que converja (reintentar si falla, como en sklearn)
print("Entrenando modelo XOR...")
max_intentos = 10
for intento in range(max_intentos):
    tf.random.set_seed(intento)  # Probar diferentes semillas
    model_xor_tf = crear_modelo_xor()
    history_xor = model_xor_tf.fit(X_xor, y_xor, epochs=500, verbose=0)
    
    predictions_xor = model_xor_tf.predict(X_xor, verbose=0)
    final_acc = (np.round(predictions_xor.flatten()) == y_xor.flatten()).mean()
    
    if final_acc == 1.0:
        print(f"✅ Convergió en el intento {intento + 1} (semilla={intento})")
        break
else:
    print(f"⚠️ No convergió al 100% tras {max_intentos} intentos")

print("\n" + "=" * 50)
print("PROBLEMA XOR CON TENSORFLOW")
print("=" * 50)
print("\nEntrada\t\tEsperado\tPredicción")
print("-" * 50)
for i in range(4):
    pred = predictions_xor[i][0]
    pred_class = 1 if pred > 0.5 else 0
    check = "✅" if pred_class == y_xor[i][0] else "❌"
    print(f"{X_xor[i]}\t\t{int(y_xor[i][0])}\t\t{pred:.4f} → {pred_class} {check}")

print(f"\n✅ Precisión final: {final_acc*100:.0f}%")

In [ ]:
# Visualizar la superficie de decisión de TensorFlow para XOR
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Crear malla
resolution = 100
x1_range = np.linspace(-0.5, 1.5, resolution)
x2_range = np.linspace(-0.5, 1.5, resolution)
X1_grid, X2_grid = np.meshgrid(x1_range, x2_range)
X_grid_flat = np.c_[X1_grid.ravel(), X2_grid.ravel()].astype(np.float32)

# Predicciones
Z_proba_xor = model_xor_tf.predict(X_grid_flat, verbose=0).reshape(X1_grid.shape)

# --- Gráfico 1: Superficie de probabilidad ---
from mpl_toolkits.mplot3d import Axes3D
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
surf = ax1.plot_surface(X1_grid, X2_grid, Z_proba_xor, cmap='coolwarm', alpha=0.8)
ax1.scatter(X_xor[:, 0], X_xor[:, 1], y_xor.flatten(), 
            c=['blue' if yi == 0 else 'red' for yi in y_xor.flatten()], 
            s=200, edgecolor='black', linewidth=2)
ax1.set_xlabel('X1')
ax1.set_ylabel('X2')
ax1.set_zlabel('P(Clase 1)')
ax1.set_title('Superficie de Decisión XOR (TensorFlow)', fontsize=12)
ax1.view_init(elev=25, azim=45)

# --- Gráfico 2: Vista 2D con frontera ---
ax2 = fig.add_subplot(1, 2, 2)
contour = ax2.contourf(X1_grid, X2_grid, Z_proba_xor, levels=20, cmap='coolwarm', alpha=0.8)
ax2.contour(X1_grid, X2_grid, Z_proba_xor, levels=[0.5], colors='green', linewidths=3)
cmap_bold = ListedColormap(['#0000FF', '#FF0000'])
ax2.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor.flatten(), cmap=cmap_bold, 
            s=200, edgecolor='black', linewidth=2)
ax2.set_xlabel('X1')
ax2.set_ylabel('X2')
ax2.set_title('Frontera de Decisión XOR (TensorFlow)', fontsize=12)
fig.colorbar(contour, ax=ax2, label='P(Clase 1)')

plt.suptitle('TensorFlow también resuelve XOR con 2 neuronas ocultas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🎯 El resultado es equivalente a scikit-learn, pero con más control y opciones.")